# 11. Event-centric Multi-horizon Entry V7

모든 확정봉을 후보로 보존하고 최근 10봉의 순서형 OHLC 특징과 20/60분 context를 결합한다. Quote는 `last_ask → 미래 last_bid` 라벨에만 사용하며 모델 입력에는 포함하지 않는다.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENT.md').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.event_centric_entry import run_event_centric_experiment

assert 'envs/urban' in str(Path(sys.executable).resolve()), sys.executable
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 160)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'python: {sys.executable}')

PROJECT_ROOT: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
python: /home/user/anaconda3/envs/urban/bin/python


## 1. 데이터 재구성과 학습

Primary는 `+3%/3분`이다. `+1%/1분`, `+2%/2분`과 미래 최대·최소 bid 수익률을 보조 target으로 학습한다. 양성·hard-negative cluster는 총 weight가 1이 되게 줄이고, 나머지 음성은 종목별 최소 3분 간격으로 sampling한다. Epoch과 decision head는 Test가 아니라 Train 날짜 walk-forward OOF로 고른다.

In [2]:
result = run_event_centric_experiment(PROJECT_ROOT)
print(f"device: {result['device']}")
print(f"parameters: {result['parameter_count']:,}")
print(f"selected epoch/head: {result['selected_epoch']} / {result['selected_score_source']}")
print(f"positive class weights: {result['positive_weights']}")
print(f"exclusions: {result['exclusions']}")

device: cuda
parameters: 5,285
selected epoch/head: 12 / head_up_1pct_1m
positive class weights: [5. 5. 5.]
exclusions: {'invalid_current_quote': 214, 'invalid_future_quote': 534, 'insufficient_future': 598}


## 2. 표본과 event sampling

In [3]:
display(result['sampling_balance'])
display(result['epoch_selection'])

,session,candidate_rows,sampled_rows,positives,hard_negatives,bearish_rows
0,session_2026-07-07,2722,1060,70,110,1208
1,session_2026-07-08,7133,3024,279,504,3209
2,session_2026-07-09,6788,2762,223,378,3049
3,session_2026-07-10,6925,3147,521,510,3181
4,session_2026-07-13,6015,2539,284,381,2802
5,session_2026-07-14,4697,1982,206,299,2230
6,session_2026-07-15,6746,2887,312,473,3195
7,session_2026-07-16,4427,1893,200,319,1984
8,session_2026-07-17,4211,1611,67,173,1842


,epoch,score_source,oof_primary_pr_auc,oof_primary_pr_lift,oof_bearish_pr_auc,oof_bearish_pr_lift
0,12,head_up_1pct_1m,0.165791,3.055538,0.183551,3.282052
1,16,head_up_1pct_1m,0.165786,3.055453,0.182509,3.263424
2,20,head_up_1pct_1m,0.165612,3.052236,0.183713,3.284952
3,20,head_up_2pct_2m,0.163396,3.011395,0.182969,3.271643
4,20,mean_all_heads,0.161539,2.977184,0.182120,3.256460
5,12,mean_all_heads,0.161355,2.973789,0.180179,3.221760
6,8,head_up_1pct_1m,0.161162,2.970226,0.177008,3.165063
7,16,head_up_2pct_2m,0.160998,2.967209,0.178035,3.183425
8,12,head_up_2pct_2m,0.160912,2.965619,0.177434,3.172671
9,16,mean_all_heads,0.160559,2.959123,0.179513,3.209850


## 3. OOF와 고정 Test

`decision_for_up_3pct_3m`은 OOF에서 선택된 multi-horizon head로 +3%/3분을 순위화한 결과다. 전체 봉과 기존 전략과 동일한 하락봉 subgroup을 함께 표시한다.

In [4]:
decision_target = 'decision_for_up_3pct_3m'
decision_metrics = result['metrics'].query("target == @decision_target and evaluation_group in ['oof', 'test']")
display(decision_metrics)
display(result['threshold'])
display(result['threshold_summary'])

,evaluation_group,subgroup,session,target,samples,positives,positive_rate,pr_auc,pr_lift,roc_auc,mean_score
96,oof,all_candles,ALL,decision_for_up_3pct_3m,24383,1323,0.054259,0.165791,3.055538,0.797802,0.265399
97,oof,all_candles,session_2026-07-10,decision_for_up_3pct_3m,6925,521,0.075235,0.181052,2.406492,0.768509,0.298346
98,oof,all_candles,session_2026-07-13,decision_for_up_3pct_3m,6015,284,0.047215,0.187659,3.974528,0.822862,0.239270
99,oof,all_candles,session_2026-07-14,decision_for_up_3pct_3m,4697,206,0.043858,0.163386,3.725362,0.826143,0.244750
100,oof,all_candles,session_2026-07-15,decision_for_up_3pct_3m,6746,312,0.046250,0.148362,3.207843,0.777033,0.269251
101,oof,bearish_only,ALL,decision_for_up_3pct_3m,11408,638,0.055926,0.183551,3.282052,0.806187,0.283579
102,oof,bearish_only,session_2026-07-10,decision_for_up_3pct_3m,3181,256,0.080478,0.205496,2.553450,0.784736,0.322347
103,oof,bearish_only,session_2026-07-13,decision_for_up_3pct_3m,2802,130,0.046395,0.202761,4.370285,0.816945,0.247095
104,oof,bearish_only,session_2026-07-14,decision_for_up_3pct_3m,2230,100,0.044843,0.182269,4.064610,0.826657,0.267596
105,oof,bearish_only,session_2026-07-15,decision_for_up_3pct_3m,3195,152,0.047574,0.179122,3.765107,0.798478,0.288131


,threshold,method,oof_score_quantile,selected_epoch,selected_score_source,test_outcome_used
0,0.597457,oof_score_top_5pct,0.95,12,head_up_1pct_1m,False


,evaluation_group,subgroup,threshold,samples,signals,signal_share,true_positives,precision,recall,actual_positive_clusters,predicted_signal_clusters,captured_positive_clusters
0,oof,all_candles,0.597457,24383,1220,0.050035,256,0.209836,0.193500,487,255,147
1,oof,bearish_only,0.597457,11408,918,0.080470,194,0.211329,0.304075,453,279,146
2,test,all_candles,0.597457,8638,141,0.016323,11,0.078014,0.041199,103,50,10
3,test,bearish_only,0.597457,3826,115,0.030058,9,0.078261,0.072581,96,51,9


## 4. V5/V6 비교

V5/V6는 하락봉 전용이므로 V7의 `bearish_only`와 직접 비교한다. V7 `all_candles`는 새 전략 범위로 별도 표시한다. 모든 Test 결과는 이미 반복 확인한 non-pristine 진단값이다.

In [5]:
data_root = (PROJECT_ROOT / result['config']['data']['root']).resolve()
processed = data_root / 'processed'
comparison_rows = []
for label, version in [
    ('V5 bearish / 15 epoch', 'quote_surge_bearish_3m_up3_binary_v5'),
    ('V6 bearish / weighted 40 epoch', 'quote_surge_bearish_3m_up3_binary_weighted40_v6'),
]:
    metrics = pd.read_parquet(processed / f'{version}_metrics.parquet').query("evaluation_group == 'test' and session == 'ALL'").iloc[0]
    threshold = pd.read_parquet(processed / f'{version}_threshold_summary.parquet').query("evaluation_group == 'test'").iloc[0]
    comparison_rows.append({'model': label, 'candidate': 'bearish_only', 'test_pr_auc': metrics['pr_auc'], 'precision': threshold['precision'], 'recall': threshold['recall'], 'signals': int(threshold['signals'])})
for subgroup in ['bearish_only', 'all_candles']:
    metrics = decision_metrics.query("evaluation_group == 'test' and subgroup == @subgroup and session == 'ALL'").iloc[0]
    threshold = result['threshold_summary'].query("evaluation_group == 'test' and subgroup == @subgroup").iloc[0]
    comparison_rows.append({'model': 'V7 event-centric', 'candidate': subgroup, 'test_pr_auc': metrics['pr_auc'], 'precision': threshold['precision'], 'recall': threshold['recall'], 'signals': int(threshold['signals'])})
comparison = pd.DataFrame(comparison_rows)
display(comparison)

,model,candidate,test_pr_auc,precision,recall,signals
0,V5 bearish / 15 epoch,bearish_only,0.120981,0.241379,0.112903,58
1,V6 bearish / weighted 40 epoch,bearish_only,0.114699,0.189655,0.088710,58
2,V7 event-centric,bearish_only,0.087760,0.078261,0.072581,115
3,V7 event-centric,all_candles,0.095349,0.078014,0.041199,141


## 5. 결론

In [6]:
oof = decision_metrics.query("evaluation_group == 'oof' and subgroup == 'all_candles' and session == 'ALL'").iloc[0]
test = decision_metrics.query("evaluation_group == 'test' and subgroup == 'all_candles' and session == 'ALL'").iloc[0]
selected = result['threshold_summary'].query("evaluation_group == 'test' and subgroup == 'all_candles'").iloc[0]
display(Markdown(
    f"**V7 OOF/Test PR-AUC {oof['pr_auc']:.3f}/{test['pr_auc']:.3f}. "
    f"OOF 고정 threshold의 Test 신호 {int(selected['signals'])}건, 적중 {int(selected['true_positives'])}건, "
    f"precision {selected['precision']:.1%}, recall {selected['recall']:.1%}.**"
))
display(Markdown('진입 모델이 배포 기준을 충족하지 못했으므로 이 신호를 전제로 한 Exit AI 학습은 진행하지 않는다. 먼저 더 많은 독립 날짜와 regime 일반화가 필요하다.'))

**V7 OOF/Test PR-AUC 0.166/0.095. OOF 고정 threshold의 Test 신호 141건, 적중 11건, precision 7.8%, recall 4.1%.**

진입 모델이 배포 기준을 충족하지 못했으므로 이 신호를 전제로 한 Exit AI 학습은 진행하지 않는다. 먼저 더 많은 독립 날짜와 regime 일반화가 필요하다.

## 6. 저장 artifact

In [7]:
display(pd.DataFrame({'artifact': result['paths'].keys(), 'path': map(str, result['paths'].values())}))

,artifact,path
0,labels,/home/user/urbandatalab/YSLee/data/stock_data/...
1,metadata,/home/user/urbandatalab/YSLee/data/stock_data/...
2,features,/home/user/urbandatalab/YSLee/data/stock_data/...
3,schema,/home/user/urbandatalab/YSLee/data/stock_data/...
4,predictions,/home/user/urbandatalab/YSLee/data/stock_data/...
5,metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
6,epoch_selection,/home/user/urbandatalab/YSLee/data/stock_data/...
7,threshold,/home/user/urbandatalab/YSLee/data/stock_data/...
8,threshold_summary,/home/user/urbandatalab/YSLee/data/stock_data/...
9,sampling_balance,/home/user/urbandatalab/YSLee/data/stock_data/...
